In [1]:
import pandas as pd
import numpy as np

In [2]:
movies = pd.read_csv("../data/ml-32m/movies.csv")

In [3]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [4]:
movies.shape

(87585, 3)

In [5]:
movies.info()

<class 'pandas.DataFrame'>
RangeIndex: 87585 entries, 0 to 87584
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   movieId  87585 non-null  int64
 1   title    87585 non-null  str  
 2   genres   87585 non-null  str  
dtypes: int64(1), str(2)
memory usage: 2.0 MB


In [6]:
movies.isnull().sum()

movieId    0
title      0
genres     0
dtype: int64

In [7]:
ratings = pd.read_csv(
    "../data/ml-32m/ratings.csv"
)

In [8]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,17,4.0,944249077
1,1,25,1.0,944250228
2,1,29,2.0,943230976
3,1,30,5.0,944249077
4,1,32,5.0,943228858


In [9]:
ratings.shape

(32000204, 4)

In [10]:
ratings.info()

<class 'pandas.DataFrame'>
RangeIndex: 32000204 entries, 0 to 32000203
Data columns (total 4 columns):
 #   Column     Dtype  
---  ------     -----  
 0   userId     int64  
 1   movieId    int64  
 2   rating     float64
 3   timestamp  int64  
dtypes: float64(1), int64(3)
memory usage: 976.6 MB


In [11]:
ratings["rating"].describe()

count    3.200020e+07
mean     3.540396e+00
std      1.058986e+00
min      5.000000e-01
25%      3.000000e+00
50%      3.500000e+00
75%      4.000000e+00
max      5.000000e+00
Name: rating, dtype: float64

In [12]:
movies["genres"].head(10)

0    Adventure|Animation|Children|Comedy|Fantasy
1                     Adventure|Children|Fantasy
2                                 Comedy|Romance
3                           Comedy|Drama|Romance
4                                         Comedy
5                          Action|Crime|Thriller
6                                 Comedy|Romance
7                             Adventure|Children
8                                         Action
9                      Action|Adventure|Thriller
Name: genres, dtype: str

In [13]:
ratings["rating"].value_counts().sort_index()

rating
0.5     525132
1.0     946675
1.5     531063
2.0    2028622
2.5    1685386
3.0    6054990
3.5    4290105
4.0    8367654
4.5    2974000
5.0    4596577
Name: count, dtype: int64

In [14]:
ratings["rating"].mean()

np.float64(3.5403956487277393)

In [15]:
movie_stats = ratings.groupby("movieId").agg(
    avg_rating=("rating", "mean"),
    rating_count=("rating", "count")
)

movie_stats.head()

,avg_rating,rating_count
movieId,,
1,3.897438,68997
2,3.275758,28904
3,3.139447,13134
4,2.845331,2806
5,3.059602,13154


In [16]:
movie_stats.sort_values(
    "avg_rating",
    ascending=False
).head(10)

,avg_rating,rating_count
movieId,,
282081,5.0,1
282073,5.0,1
282021,5.0,1
282019,5.0,1
193529,5.0,2
193557,5.0,1
194066,5.0,1
193727,5.0,1
281956,5.0,1


In [17]:
popular_movies = movie_stats[
    movie_stats["rating_count"] >= 1000
]

In [18]:
popular_movies.sort_values(
    "avg_rating",
    ascending=False
).head(10)

,avg_rating,rating_count
movieId,,
171011,4.446830,1956
159817,4.444369,2948
170705,4.426539,2811
318,4.404614,102929
858,4.317030,66440
202439,4.312254,11670
179135,4.300086,1163
198185,4.298684,1140
1203,4.265311,21863


In [19]:
movie_stats = movie_stats.reset_index()

movie_stats = movie_stats.merge(
    movies,
    on="movieId"
)

In [20]:
movie_stats.head()

,movieId,avg_rating,rating_count,title,genres
0,1,3.897438,68997,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,3.275758,28904,Jumanji (1995),Adventure|Children|Fantasy
2,3,3.139447,13134,Grumpier Old Men (1995),Comedy|Romance
3,4,2.845331,2806,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,3.059602,13154,Father of the Bride Part II (1995),Comedy


In [21]:
def popular_movies(min_ratings=1000, n=10):
    
    filtered = movie_stats[
        movie_stats["rating_count"] >= min_ratings
    ]
    
    recommendations = filtered.sort_values(
        ["avg_rating", "rating_count"],
        ascending=[False, False]
    )
    
    return recommendations[
        ["title", "genres", "avg_rating", "rating_count"]
    ].head(n)

In [22]:
popular_movies()

,title,genres,avg_rating,rating_count
44190,Planet Earth II (2016),Documentary,4.446830,1956
39306,Planet Earth (2006),Documentary,4.444369,2948
44042,Band of Brothers (2001),Action|Drama|War,4.426539,2811
314,"Shawshank Redemption, The (1994)",Crime|Drama,4.404614,102929
840,"Godfather, The (1972)",Crime|Drama,4.317030,66440
58698,Parasite (2019),Comedy|Drama,4.312254,11670
47975,Blue Planet II (2017),Documentary,4.300086,1163
56760,Twin Peaks (1989),Drama|Mystery,4.298684,1140
1173,12 Angry Men (1957),Drama,4.265311,21863
49,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,4.265070,67750


In [23]:
movies["genres"].head()

0    Adventure|Animation|Children|Comedy|Fantasy
1                     Adventure|Children|Fantasy
2                                 Comedy|Romance
3                           Comedy|Drama|Romance
4                                         Comedy
Name: genres, dtype: str

In [24]:
genre_matrix = movies["genres"].str.get_dummies(sep="|")

In [25]:
genre_matrix.head()

,(no genres listed),Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,0,0,1,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0
3,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0
4,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [26]:
from sklearn.metrics.pairwise import cosine_similarity

In [27]:
from scipy.sparse import csr_matrix

genre_sparse = csr_matrix(genre_matrix)

In [28]:
genre_sparse.shape

(87585, 20)

In [29]:
movie_indices = pd.Series(
    movies.index,
    index=movies["title"]
)

In [30]:
movie_indices["Toy Story (1995)"]

np.int64(0)

In [31]:
movie_idx = movie_indices["Toy Story (1995)"]

movie_vector = genre_sparse[movie_idx]

In [32]:
movie_vector.shape

(1, 20)

In [33]:
from sklearn.metrics.pairwise import cosine_similarity

similarities = cosine_similarity(
    movie_vector,
    genre_sparse
)

similarities = similarities.flatten()

In [34]:
similarities.shape

(87585,)

In [35]:
similarity_scores = list(enumerate(similarities))

similarity_scores = sorted(
    similarity_scores,
    key=lambda x: x[1],
    reverse=True
)

similarity_scores[:10]

[(0, np.float64(0.9999999999999999)),
 (2203, np.float64(0.9999999999999999)),
 (3021, np.float64(0.9999999999999999)),
 (3654, np.float64(0.9999999999999999)),
 (3913, np.float64(0.9999999999999999)),
 (4781, np.float64(0.9999999999999999)),
 (9951, np.float64(0.9999999999999999)),
 (10775, np.float64(0.9999999999999999)),
 (11606, np.float64(0.9999999999999999)),
 (12970, np.float64(0.9999999999999999))]

In [36]:
key=lambda x: x[1]

In [37]:
reverse=True

In [38]:
similarity_scores = similarity_scores[1:]

In [39]:
similarity_scores[:10]

[(2203, np.float64(0.9999999999999999)),
 (3021, np.float64(0.9999999999999999)),
 (3654, np.float64(0.9999999999999999)),
 (3913, np.float64(0.9999999999999999)),
 (4781, np.float64(0.9999999999999999)),
 (9951, np.float64(0.9999999999999999)),
 (10775, np.float64(0.9999999999999999)),
 (11606, np.float64(0.9999999999999999)),
 (12970, np.float64(0.9999999999999999)),
 (17436, np.float64(0.9999999999999999))]

In [40]:
recommended_indices = [
    index
    for index, score in similarity_scores[:10]
]

movies.iloc[recommended_indices][
    ["title", "genres"]
]

,title,genres
2203,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy
3021,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy
3654,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure|Animation|Children|Comedy|Fantasy
3913,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy
4781,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy
9951,DuckTales: The Movie - Treasure of the Lost La...,Adventure|Animation|Children|Comedy|Fantasy
10775,"Wild, The (2006)",Adventure|Animation|Children|Comedy|Fantasy
11606,Shrek the Third (2007),Adventure|Animation|Children|Comedy|Fantasy
12970,"Tale of Despereaux, The (2008)",Adventure|Animation|Children|Comedy|Fantasy
17436,Asterix and the Vikings (Astérix et les Viking...,Adventure|Animation|Children|Comedy|Fantasy


In [41]:
def recommend_movies(movie_title, n=10):
    # Find the index of the selected movie
    movie_idx = movie_indices[movie_title]

    # Get the selected movie's feature vector
    movie_vector = genre_sparse[movie_idx]

    # Calculate similarity with every movie
    similarities = cosine_similarity(
        movie_vector,
        genre_sparse
    ).flatten()

    # Create (movie_index, similarity_score) pairs
    similarity_scores = list(enumerate(similarities))

    # Sort from highest similarity to lowest
    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    # Remove the selected movie itself
    similarity_scores = similarity_scores[1:n+1]

    # Get movie indexes
    recommended_indices = [
        index
        for index, score in similarity_scores
    ]

    # Return movie information
    return movies.iloc[recommended_indices][
        ["title", "genres"]
    ]

In [42]:
recommend_movies

<function __main__.recommend_movies(movie_title, n=10)>

In [43]:
recommend_movies("Toy Story (1995)", n=10)

,title,genres
2203,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy
3021,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy
3654,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure|Animation|Children|Comedy|Fantasy
3913,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy
4781,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy
9951,DuckTales: The Movie - Treasure of the Lost La...,Adventure|Animation|Children|Comedy|Fantasy
10775,"Wild, The (2006)",Adventure|Animation|Children|Comedy|Fantasy
11606,Shrek the Third (2007),Adventure|Animation|Children|Comedy|Fantasy
12970,"Tale of Despereaux, The (2008)",Adventure|Animation|Children|Comedy|Fantasy
17436,Asterix and the Vikings (Astérix et les Viking...,Adventure|Animation|Children|Comedy|Fantasy


In [44]:
movies[
    movies["title"].str.contains(
        "Dark Knight",
        case=False,
        na=False
    )
][["movieId", "title", "genres"]]

,movieId,title,genres
12223,58559,"Dark Knight, The (2008)",Action|Crime|Drama|IMAX
17469,91529,"Dark Knight Rises, The (2012)",Action|Adventure|Crime|IMAX
27906,130219,The Dark Knight (2011),Action|Crime|Drama|Thriller
72803,236067,The Fire Rises: The Creation and Impact of The...,Documentary
76553,253310,Batman: The Dark Knight Returns (2013),Action|Animation


In [45]:
def recommend_movies(movie_title, n=10):

    # Check whether the exact title exists
    if movie_title not in movie_indices:

        matches = movies[
            movies["title"].str.contains(
                movie_title,
                case=False,
                na=False
            )
        ]

        if len(matches) == 0:
            return "Movie not found."

        return matches[
            ["movieId", "title", "genres"]
        ]

    # Find movie index
    movie_idx = movie_indices[movie_title]

    # Get movie feature vector
    movie_vector = genre_sparse[movie_idx]

    # Calculate similarity with every movie
    similarities = cosine_similarity(
        movie_vector,
        genre_sparse
    ).flatten()

    # Create (index, similarity) pairs
    similarity_scores = list(
        enumerate(similarities)
    )

    # Sort highest similarity first
    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    # Remove selected movie
    similarity_scores = similarity_scores[1:n+1]

    # Get movie indexes
    recommended_indices = [
        index
        for index, score in similarity_scores
    ]

    # Get movie information
    recommendations = movies.iloc[
        recommended_indices
    ][["title", "genres"]].copy()

    # Add similarity scores
    recommendations["similarity"] = [
        score
        for index, score in similarity_scores
    ]

    return recommendations

In [46]:
recommend_movies("Dark Knight, The (2008)", n=10)

,title,genres,similarity
21294,Need for Speed (2014),Action|Crime|Drama|IMAX,1.000000
16383,"Fast Five (Fast and the Furious 5, The) (2011)",Action|Crime|Drama|Thriller|IMAX,0.894427
41,Dead Presidents (1995),Action|Crime|Drama,0.866025
379,Bad Company (1995),Action|Crime|Drama,0.866025
385,Faster Pussycat! Kill! Kill! (1965),Action|Crime|Drama,0.866025
488,Menace II Society (1993),Action|Crime|Drama,0.866025
679,"Substitute, The (1996)",Action|Crime|Drama,0.866025
856,Nothing to Lose (1994),Action|Crime|Drama,0.866025
2104,"Untouchables, The (1987)",Action|Crime|Drama,0.866025
2190,Monument Ave. (1998),Action|Crime|Drama,0.866025


In [47]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,17,4.0,944249077
1,1,25,1.0,944250228
2,1,29,2.0,943230976
3,1,30,5.0,944249077
4,1,32,5.0,943228858


In [48]:
ratings_sample = ratings.head(10000)

user_item_matrix = ratings_sample.pivot_table(
    index="userId",
    columns="movieId",
    values="rating"
)

user_item_matrix.shape

(66, 3965)

In [49]:
user_item_matrix.head()

movieId,1,2,3,4,5,6,7,8,10,11,...,278702,279412,280236,280258,280868,285593,286897,287699,288513,291485
userId,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,3.5,NaN,NaN,NaN,NaN,NaN,NaN,4.0,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [50]:
total_values = user_item_matrix.size

missing_values = user_item_matrix.isna().sum().sum()

sparsity = missing_values / total_values

sparsity

np.float64(0.9617868470327486)

In [51]:
user_item_filled = user_item_matrix.fillna(0)

user_item_filled.head()

movieId,1,2,3,4,5,6,7,8,10,11,...,278702,279412,280236,280258,280868,285593,286897,287699,288513,291485
userId,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,3.5,0.0,0.0,0.0,0.0,0.0,0.0,4.0,4.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [52]:
from sklearn.metrics.pairwise import cosine_similarity

user_similarity = cosine_similarity(
    user_item_filled
)

user_similarity.shape

(66, 66)

In [53]:
user_similarity[:5, :5]

array([[1.        , 0.01375202, 0.10452492, 0.05668801, 0.0370296 ],
       [0.01375202, 1.        , 0.2489614 , 0.        , 0.35153975],
       [0.10452492, 0.2489614 , 1.        , 0.03017519, 0.35805258],
       [0.05668801, 0.        , 0.03017519, 1.        , 0.        ],
       [0.0370296 , 0.35153975, 0.35805258, 0.        , 1.        ]])

In [54]:
user_idx = 0

similar_users = list(
    enumerate(user_similarity[user_idx])
)

similar_users = sorted(
    similar_users,
    key=lambda x: x[1],
    reverse=True
)

similar_users[:10]

[(0, np.float64(1.0000000000000002)),
 (39, np.float64(0.32640640756122496)),
 (58, np.float64(0.2604394296590725)),
 (64, np.float64(0.20606315178254106)),
 (27, np.float64(0.1827370825638076)),
 (19, np.float64(0.17074136540932872)),
 (32, np.float64(0.16285947101489925)),
 (49, np.float64(0.1617710378700152)),
 (41, np.float64(0.158283457635649)),
 (34, np.float64(0.14374746603806593))]

In [55]:
similar_users = similar_users[1:]

In [56]:
similar_users[:10]

[(39, np.float64(0.32640640756122496)),
 (58, np.float64(0.2604394296590725)),
 (64, np.float64(0.20606315178254106)),
 (27, np.float64(0.1827370825638076)),
 (19, np.float64(0.17074136540932872)),
 (32, np.float64(0.16285947101489925)),
 (49, np.float64(0.1617710378700152)),
 (41, np.float64(0.158283457635649)),
 (34, np.float64(0.14374746603806593)),
 (55, np.float64(0.132991269984422))]

In [57]:
similar_user_indices = [
    index
    for index, score in similar_users[:10]
]

similar_user_ids = user_item_filled.index[
    similar_user_indices
]

similar_user_ids

Index([40, 59, 65, 28, 20, 33, 50, 42, 35, 56], dtype='int64', name='userId')

In [58]:
similar_user_ratings = ratings_sample[
    ratings_sample["userId"].isin(similar_user_ids)
]

liked_movies = similar_user_ratings[
    similar_user_ratings["rating"] >= 4
]

liked_movies.head()

,userId,movieId,rating,timestamp
2004,20,1,5.0,1553184230
2005,20,32,4.0,1553187324
2006,20,47,4.0,1553187333
2007,20,50,4.5,1553180577
2010,20,260,5.0,1553180585


In [59]:
current_user_id = user_item_filled.index[user_idx]

watched_movies = ratings_sample[
    ratings_sample["userId"] == current_user_id
]["movieId"]

watched_movies.head()

0    17
1    25
2    29
3    30
4    32
Name: movieId, dtype: int64

In [60]:
candidate_movies = liked_movies[
    ~liked_movies["movieId"].isin(watched_movies)
]

candidate_movies.head()

,userId,movieId,rating,timestamp
2004,20,1,5.0,1553184230
2006,20,47,4.0,1553187333
2007,20,50,4.5,1553180577
2011,20,296,5.0,1553187334
2012,20,318,5.0,1553180573


In [61]:
similar_users_df = pd.DataFrame(
    similar_users[:10],
    columns=["user_index", "similarity"]
)

similar_users_df

,user_index,similarity
0,39,0.326406
1,58,0.260439
2,64,0.206063
3,27,0.182737
4,19,0.170741
5,32,0.162859
6,49,0.161771
7,41,0.158283
8,34,0.143747
9,55,0.132991


In [62]:
similar_users_df["userId"] = user_item_filled.index[
    similar_users_df["user_index"]
]

similar_users_df

,user_index,similarity,userId
0,39,0.326406,40
1,58,0.260439,59
2,64,0.206063,65
3,27,0.182737,28
4,19,0.170741,20
5,32,0.162859,33
6,49,0.161771,50
7,41,0.158283,42
8,34,0.143747,35
9,55,0.132991,56


In [63]:
similar_user_ratings = similar_user_ratings.merge(
    similar_users_df[["userId", "similarity"]],
    on="userId",
    how="inner"
)

similar_user_ratings.head()

,userId,movieId,rating,timestamp,similarity
0,20,1,5.0,1553184230,0.170741
1,20,32,4.0,1553187324,0.170741
2,20,47,4.0,1553187333,0.170741
3,20,50,4.5,1553180577,0.170741
4,20,110,3.0,1553184216,0.170741


In [64]:
similar_user_ratings["weighted_score"] = (
    similar_user_ratings["similarity"]
    * similar_user_ratings["rating"]
)

similar_user_ratings.head()

,userId,movieId,rating,timestamp,similarity,weighted_score
0,20,1,5.0,1553184230,0.170741,0.853707
1,20,32,4.0,1553187324,0.170741,0.682965
2,20,47,4.0,1553187333,0.170741,0.682965
3,20,50,4.5,1553180577,0.170741,0.768336
4,20,110,3.0,1553184216,0.170741,0.512224


In [65]:
movie_scores = similar_user_ratings.groupby("movieId").agg(
    weighted_score=("weighted_score", "sum"),
    similarity_sum=("similarity", "sum"),
    rating_count=("rating", "count")
)

In [66]:
movie_scores["score"] = (
    movie_scores["weighted_score"]
    / movie_scores["similarity_sum"]
)

In [67]:
movie_scores = movie_scores[
    ~movie_scores.index.isin(watched_movies)
]

In [68]:
top_movies = movie_scores.sort_values(
    "score",
    ascending=False
).head(10)

top_movies

,weighted_score,similarity_sum,rating_count,score
movieId,,,,
1243,1.632423,0.326485,2,5.0
2248,2.215883,0.443177,2,5.0
148626,1.632423,0.326485,2,5.0
3508,0.913685,0.182737,1,5.0
270698,0.913685,0.182737,1,5.0
272417,0.913685,0.182737,1,5.0
208703,0.913685,0.182737,1,5.0
209315,0.913685,0.182737,1,5.0
212361,0.913685,0.182737,1,5.0


In [69]:
top_movies = top_movies.reset_index()

top_movies = top_movies.merge(
    movies[["movieId", "title", "genres"]],
    on="movieId",
    how="left"
)

top_movies[
    ["title", "genres", "score", "rating_count"]
]

,title,genres,score,rating_count
0,Rosencrantz and Guildenstern Are Dead (1990),Comedy|Drama,5.0,2
1,Say Anything... (1989),Comedy|Drama|Romance,5.0,2
2,"Big Short, The (2015)",Drama,5.0,2
3,"Outlaw Josey Wales, The (1976)",Action|Adventure|Drama|Thriller|Western,5.0,1
4,Everything Everywhere All at Once (2022),Action|Comedy|Sci-Fi,5.0,1
5,RRR (2022),Action|Drama|War,5.0,1
6,1917 (2019),Drama|War,5.0,1
7,Little Women (2019),Drama|Romance,5.0,1
8,Sound of Metal (2019),Drama,5.0,1
9,On the Town (1949),Comedy|Musical|Romance,5.0,1


In [70]:
current_user_ratings = ratings[
    ratings["userId"] == current_user_id
].sort_values("timestamp")

In [71]:
current_user_ratings.shape

(141, 4)

In [72]:
split_point = int(
    len(current_user_ratings) * 0.8
)

train_ratings = current_user_ratings.iloc[
    :split_point
]

test_ratings = current_user_ratings.iloc[
    split_point:
]

In [73]:
print("Training ratings:", len(train_ratings))
print("Test ratings:", len(test_ratings))

Training ratings: 112
Test ratings: 29


In [74]:
relevant_test_movies = set(
    test_ratings[
        test_ratings["rating"] >= 4
    ]["movieId"]
)

len(relevant_test_movies)


20

In [75]:
train_ratings_all = ratings[
    ~(
        (ratings["userId"] == current_user_id)
        &
        (ratings["movieId"].isin(
            test_ratings["movieId"]
        ))
    )
]

In [76]:
user_train_ratings = train_ratings_all[
    train_ratings_all["userId"] == current_user_id
]

In [77]:
user_train_ratings.shape

(112, 4)

In [78]:
watched_train_movies = set(
    user_train_ratings["movieId"]
)

In [79]:
evaluation_scores = movie_scores[
    ~movie_scores.index.isin(watched_train_movies)
].copy()

In [80]:
top_10 = evaluation_scores.sort_values(
    "score",
    ascending=False
).head(10)

recommended_movie_ids = set(
    top_10.index
)

recommended_movie_ids

{1243, 2248, 3508, 3606, 148626, 208703, 209315, 212361, 270698, 272417}

In [81]:
hits = (
    recommended_movie_ids
    & relevant_test_movies
)

precision_at_10 = (
    len(hits) / 10
)

recall_at_10 = (
    len(hits) / len(relevant_test_movies)
)

print("Hits:", len(hits))
print("Precision@10:", precision_at_10)
print("Recall@10:", recall_at_10)

Hits: 0
Precision@10: 0.0
Recall@10: 0.0


In [82]:
user_rating_counts = ratings.groupby("userId").size()

user_rating_counts.describe()

count    200948.000000
mean        159.246193
std         282.025462
min          20.000000
25%          36.000000
50%          73.000000
75%         167.000000
max       33332.000000
dtype: float64

In [83]:
eligible_users = user_rating_counts[
    user_rating_counts >= 50
].index

len(eligible_users)

128344

In [84]:
current_user_id = eligible_users[0]

current_user_id

np.int64(1)

In [85]:
current_user_ratings = ratings[
    ratings["userId"] == current_user_id
].sort_values("timestamp")

In [86]:
len(current_user_ratings)

141

In [87]:
split_point = int(
    len(current_user_ratings) * 0.8
)

train_ratings = current_user_ratings.iloc[
    :split_point
]

test_ratings = current_user_ratings.iloc[
    split_point:
]

In [88]:
relevant_test_movies = set(
    test_ratings[
        test_ratings["rating"] >= 4
    ]["movieId"]
)

len(relevant_test_movies)

20

In [89]:
from scipy.sparse import csr_matrix

user_ids = ratings["userId"].unique()
movie_ids = ratings["movieId"].unique()

user_id_to_index = {
    user_id: i
    for i, user_id in enumerate(user_ids)
}

movie_id_to_index = {
    movie_id: i
    for i, movie_id in enumerate(movie_ids)
}

In [90]:
user_indices = ratings["userId"].map(
    user_id_to_index
).values

movie_indices_full = ratings["movieId"].map(
    movie_id_to_index
).values

rating_values = ratings["rating"].values

In [91]:
user_item_sparse = csr_matrix(
    (
        rating_values,
        (user_indices, movie_indices_full)
    ),
    shape=(
        len(user_ids),
        len(movie_ids)
    )
)

In [92]:
user_item_sparse.shape

(200948, 84432)

In [93]:
user_item_sparse.nnz

32000204

In [94]:
target_user_id = current_user_id

target_user_index = user_id_to_index[
    target_user_id
]

target_user_index

0

In [95]:
target_user_vector = user_item_sparse[
    target_user_index
]
target_user_vector.shape

(1, 84432)

In [96]:
from sklearn.metrics.pairwise import cosine_similarity

user_similarities = cosine_similarity(
    target_user_vector,
    user_item_sparse
).flatten()
user_similarities.shape

(200948,)

In [97]:
cosine_similarity(
    target_user_vector,
    user_item_sparse
)

array([[1.        , 0.01375202, 0.10452492, ..., 0.04326014, 0.060518  ,
        0.05577832]], shape=(1, 200948))

In [98]:
similar_user_indices = user_similarities.argsort()[::-1]

In [99]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [100]:
from sklearn.metrics.pairwise import cosine_similarity

In [101]:
user_similarities = cosine_similarity(
    target_user_vector,
    user_item_sparse
).flatten()

In [102]:
user_similarities.shape

(200948,)

In [103]:
similar_user_indices = user_similarities.argsort()[::-1]

similar_user_indices = similar_user_indices[
    similar_user_indices != target_user_index
]

top_similar_indices = similar_user_indices[:10]

similar_users_full = pd.DataFrame({
    "userId": user_ids[top_similar_indices],
    "similarity": user_similarities[top_similar_indices]
})

similar_users_full

,userId,similarity
0,65854,0.375467
1,60615,0.371499
2,30722,0.371321
3,95589,0.362770
4,111217,0.361920
5,93338,0.361168
6,145035,0.360779
7,33143,0.357475
8,165223,0.355825
9,164986,0.354638


In [104]:
similar_users_full

,userId,similarity
0,65854,0.375467
1,60615,0.371499
2,30722,0.371321
3,95589,0.362770
4,111217,0.361920
5,93338,0.361168
6,145035,0.360779
7,33143,0.357475
8,165223,0.355825
9,164986,0.354638


In [105]:
similar_user_ids = similar_users_full["userId"].values

similar_user_ids

array([ 65854,  60615,  30722,  95589, 111217,  93338, 145035,  33143,
       165223, 164986])

In [106]:
similar_user_ratings = ratings[
    ratings["userId"].isin(similar_user_ids)
]

similar_user_ratings.head()

,userId,movieId,rating,timestamp
4925325,30722,17,3.0,938801185
4925326,30722,21,4.0,938801812
4925327,30722,30,2.0,940999943
4925328,30722,32,2.0,938797199
4925329,30722,34,4.0,940995941


In [107]:
liked_movies = similar_user_ratings[
    similar_user_ratings["rating"] >= 4
]

liked_movies.head(
    
)

,userId,movieId,rating,timestamp
4925326,30722,21,4.0,938801812
4925329,30722,34,4.0,940995941
4925330,30722,47,4.0,941000091
4925331,30722,50,5.0,938797199
4925333,30722,111,5.0,938801601


In [108]:
watched_movies = ratings[
    ratings["userId"] == target_user_id
]["movieId"]
candidate_movies = liked_movies[
    ~liked_movies["movieId"].isin(watched_movies)
]

candidate_movies.head()

,userId,movieId,rating,timestamp
4925326,30722,21,4.0,938801812
4925330,30722,47,4.0,941000091
4925331,30722,50,5.0,938797199
4925335,30722,187,4.0,940999845
4925336,30722,194,5.0,940996263


In [109]:
similar_user_ratings = similar_user_ratings.merge(
    similar_users_full,
    on="userId",
    how="inner"
)
similar_user_ratings.head()

,userId,movieId,rating,timestamp,similarity
0,30722,17,3.0,938801185,0.371321
1,30722,21,4.0,938801812,0.371321
2,30722,30,2.0,940999943,0.371321
3,30722,32,2.0,938797199,0.371321
4,30722,34,4.0,940995941,0.371321


In [111]:
similar_user_ratings["weighted_score"] = (
    similar_user_ratings["similarity_x"]
    * similar_user_ratings["rating"]
)

KeyError: 'similarity_x'

In [ ]:
movie_scores = similar_user_ratings.groupby(
    "movieId"
).agg(
    weighted_score=("weighted_score", "sum"),
    similarity_sum=("similarity_x", "sum"),
    rating_count=("rating", "count")
)
movie_scores["score"] = (
    movie_scores["weighted_score"]
    / movie_scores["similarity_sum"]
)
movie_scores = movie_scores[
    ~movie_scores.index.isin(watched_movies)
]

In [ ]:
movie_scores.sort_values(
    "score",
    ascending=False
).head(10)

,weighted_score,similarity_sum,rating_count,score
movieId,,,,
1927,3.644873,0.728975,2,5.0
28,1.805839,0.361168,1,5.0
1249,1.787375,0.357475,1,5.0
1257,1.773190,0.354638,1,5.0
73,1.779127,0.355825,1,5.0
3429,1.773190,0.354638,1,5.0
3543,1.803895,0.360779,1,5.0
2948,1.787375,0.357475,1,5.0
1161,1.877337,0.375467,1,5.0


In [112]:
top_collaborative = movie_scores.sort_values(
    "score",
    ascending=False
).head(10)

top_collaborative

,weighted_score,similarity_sum,rating_count,score
movieId,,,,
1243,1.632423,0.326485,2,5.0
2248,2.215883,0.443177,2,5.0
148626,1.632423,0.326485,2,5.0
3508,0.913685,0.182737,1,5.0
270698,0.913685,0.182737,1,5.0
272417,0.913685,0.182737,1,5.0
208703,0.913685,0.182737,1,5.0
209315,0.913685,0.182737,1,5.0
212361,0.913685,0.182737,1,5.0


In [113]:
top_collaborative = top_collaborative.reset_index()

top_collaborative = top_collaborative.merge(
    movies[["movieId", "title", "genres"]],
    on="movieId",
    how="left"
)

In [114]:
top_collaborative[
    ["movieId", "title", "genres", "score", "rating_count"]
]

,movieId,title,genres,score,rating_count
0,1243,Rosencrantz and Guildenstern Are Dead (1990),Comedy|Drama,5.0,2
1,2248,Say Anything... (1989),Comedy|Drama|Romance,5.0,2
2,148626,"Big Short, The (2015)",Drama,5.0,2
3,3508,"Outlaw Josey Wales, The (1976)",Action|Adventure|Drama|Thriller|Western,5.0,1
4,270698,Everything Everywhere All at Once (2022),Action|Comedy|Sci-Fi,5.0,1
5,272417,RRR (2022),Action|Drama|War,5.0,1
6,208703,1917 (2019),Drama|War,5.0,1
7,209315,Little Women (2019),Drama|Romance,5.0,1
8,212361,Sound of Metal (2019),Drama,5.0,1
9,3606,On the Town (1949),Comedy|Musical|Romance,5.0,1


In [115]:
def recommend_collaborative(user_id, n=10):
    
    user_index = user_id_to_index[user_id]
    
    user_vector = user_item_sparse[user_index]
    
    similarities = cosine_similarity(
        user_vector,
        user_item_sparse
    ).flatten()
    
    similar_indices = similarities.argsort()[::-1]
    
    similar_indices = similar_indices[
        similar_indices != user_index
    ]
    
    top_indices = similar_indices[:10]
    
    similar_users = pd.DataFrame({
        "userId": user_ids[top_indices],
        "similarity": similarities[top_indices]
    })
    
    similar_ratings = ratings[
        ratings["userId"].isin(
            similar_users["userId"]
        )
    ].merge(
        similar_users,
        on="userId",
        how="inner"
    )
    
    similar_ratings = similar_ratings[
        similar_ratings["rating"] >= 4
    ]
    
    similar_ratings["weighted_score"] = (
        similar_ratings["rating"]
        * similar_ratings["similarity"]
    )
    
    scores = similar_ratings.groupby("movieId").agg(
        weighted_score=("weighted_score", "sum"),
        similarity_sum=("similarity", "sum"),
        rating_count=("rating", "count")
    )
    
    scores["score"] = (
        scores["weighted_score"]
        / scores["similarity_sum"]
    )
    
    watched = ratings[
        ratings["userId"] == user_id
    ]["movieId"]
    
    scores = scores[
        ~scores.index.isin(watched)
    ]
    
    recommendations = (
        scores
        .sort_values("score", ascending=False)
        .head(n)
        .reset_index()
        .merge(
            movies[["movieId", "title", "genres"]],
            on="movieId",
            how="left"
        )
    )
    
    return recommendations[
        ["movieId", "title", "genres", "score", "rating_count"]
    ]

In [116]:
recommend_collaborative(
    target_user_id,
    n=10
)

,movieId,title,genres,score,rating_count
0,162,Crumb (1994),Documentary,5.0,3
1,1927,All Quiet on the Western Front (1930),Action|Drama|War,5.0,2
2,28,Persuasion (1995),Drama|Romance,5.0,1
3,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,5.0,2
4,3035,Mister Roberts (1955),Comedy|Drama|War,5.0,1
5,2859,Stop Making Sense (1984),Documentary|Musical,5.0,1
6,2948,From Russia with Love (1963),Action|Adventure|Thriller,5.0,1
7,3578,Gladiator (2000),Action|Adventure|Drama,5.0,1
8,2670,Run Silent Run Deep (1958),War,5.0,1
9,2628,Star Wars: Episode I - The Phantom Menace (1999),Action|Adventure|Sci-Fi,5.0,2


In [117]:
split_point = int(len(current_user_ratings) * 0.8)

train_ratings = current_user_ratings.iloc[:split_point]
test_ratings = current_user_ratings.iloc[split_point:]

In [127]:
relevant_movies = set(
    test_ratings[
        test_ratings["rating"] >= 4
    ]["movieId"]
)

len(relevant_movies)

20

In [128]:
recommendations = recommend_collaborative(
    target_user_id,
    n=10
)

In [129]:
recommended_movies = set(
    recommendations["movieId"]
)

In [131]:
precision_at_10 = (
    len(hits) / len(recommended_movies)
)

recall_at_10 = (
    len(hits) / len(relevant_movies)
)

print("Precision@10:", precision_at_10)
print("Recall@10:", recall_at_10)

Precision@10: 0.0
Recall@10: 0.0


In [132]:
recommendations_20 = recommend_collaborative(
    target_user_id,
    n=20
)

In [133]:
def precision_recall_at_k(
    recommended_ids,
    relevant_ids,
    k
):
    recommended_k = list(recommended_ids)[:k]

    relevant_ids = set(relevant_ids)

    hits = len(
        set(recommended_k) & relevant_ids
    )

    precision = hits / k

    recall = (
        hits / len(relevant_ids)
        if len(relevant_ids) > 0
        else 0
    )

    return precision, recall

In [134]:
recommended_ids = recommendations_20["movieId"].tolist()

p5, r5 = precision_recall_at_k(
    recommended_ids,
    relevant_movies,
    5
)

p10, r10 = precision_recall_at_k(
    recommended_ids,
    relevant_movies,
    10
)

p20, r20 = precision_recall_at_k(
    recommended_ids,
    relevant_movies,
    20
)

In [135]:
evaluation_results = pd.DataFrame({
    "K": [5, 10, 20],
    "Precision": [p5, p10, p20],
    "Recall": [r5, r10, r20]
})

evaluation_results

,K,Precision,Recall
0,5,0.0,0.0
1,10,0.0,0.0
2,20,0.0,0.0


In [136]:
train_ratings_all = ratings[
    ~(
        (ratings["userId"] == target_user_id)
        &
        (ratings["movieId"].isin(
            test_ratings["movieId"]
        ))
    )
].copy()

In [137]:
train_user_indices = train_ratings_all["userId"].map(
    user_id_to_index
).values

train_movie_indices = train_ratings_all["movieId"].map(
    movie_id_to_index
).values

train_rating_values = train_ratings_all["rating"].values

In [138]:
train_sparse = csr_matrix(
    (
        train_rating_values,
        (train_user_indices, train_movie_indices)
    ),
    shape=user_item_sparse.shape
)

In [140]:
target_train_vector = train_sparse[
    target_user_index
]
train_similarities = cosine_similarity(
    target_train_vector,
    train_sparse
).flatten()

In [141]:
train_similar_indices = (
    train_similarities.argsort()[::-1]
)

train_similar_indices = (
    train_similar_indices[
        train_similar_indices != target_user_index
    ]
)

top_train_indices = train_similar_indices[:10]

In [142]:
train_similar_users = pd.DataFrame({
    "userId": user_ids[top_train_indices],
    "similarity": train_similarities[
        top_train_indices
    ]
})

train_similar_users

,userId,similarity
0,30722,0.350091
1,165223,0.347387
2,170929,0.346815
3,164986,0.346177
4,93338,0.345459
5,60615,0.344654
6,197740,0.341912
7,83203,0.340219
8,78326,0.338758
9,111217,0.336545


In [143]:
train_similar_user_ids = train_similar_users["userId"].values

train_candidate_ratings = train_ratings_all[
    train_ratings_all["userId"].isin(train_similar_user_ids)
].merge(
    train_similar_users,
    on="userId",
    how="inner"
)

In [145]:
train_candidate_ratings = train_candidate_ratings[
    train_candidate_ratings["rating"] >= 4
].copy()

In [146]:
train_candidate_ratings["weighted_score"] = (
    train_candidate_ratings["rating"]
    * train_candidate_ratings["similarity"]
)

In [147]:
train_movie_scores = train_candidate_ratings.groupby(
    "movieId"
).agg(
    weighted_score=("weighted_score", "sum"),
    similarity_sum=("similarity", "sum"),
    rating_count=("rating", "count")
)

train_movie_scores["score"] = (
    train_movie_scores["weighted_score"]
    / train_movie_scores["similarity_sum"]
)

In [148]:
watched_train = set(
    train_ratings["movieId"]
)

train_movie_scores = train_movie_scores[
    ~train_movie_scores.index.isin(watched_train)
]

In [149]:
evaluation_recommendations = (
    train_movie_scores
    .sort_values("score", ascending=False)
    .head(20)
    .reset_index()
    .merge(
        movies[["movieId", "title", "genres"]],
        on="movieId",
        how="left"
    )
)

In [150]:
evaluation_recommendations[
    ["movieId", "title", "genres", "score"]
]

,movieId,title,genres,score
0,1537,Shall We Dance? (Shall We Dansu?) (1996),Comedy|Drama|Romance,5.0
1,1296,"Room with a View, A (1986)",Drama|Romance,5.0
2,6170,"Black Stallion, The (1979)",Adventure|Children|Drama,5.0
3,4186,"Fortune Cookie, The (1966)",Comedy|Drama|Romance,5.0
4,3469,Inherit the Wind (1960),Drama,5.0
5,3429,Creature Comforts (1989),Animation|Comedy,5.0
6,3685,Prizzi's Honor (1985),Comedy|Drama|Romance,5.0
7,3507,"Odd Couple, The (1968)",Comedy,5.0
8,3362,Dog Day Afternoon (1975),Crime|Drama,5.0
9,3328,Ghost Dog: The Way of the Samurai (1999),Crime|Drama,5.0


In [151]:
recommended_ids = set(
    evaluation_recommendations
    .head(10)["movieId"]
)

hits = recommended_ids.intersection(
    relevant_movies
)

precision_10 = len(hits) / 10

recall_10 = (
    len(hits) / len(relevant_movies)
    if len(relevant_movies) > 0
    else 0
)

print("Precision@10:", precision_10)
print("Recall@10:", recall_10)

Precision@10: 0.0
Recall@10: 0.0


In [152]:
def recommend_content(movie_title, n=10):

    matches = movies[
        movies["title"] == movie_title
    ]

    if matches.empty:
        return pd.DataFrame()

    movie_index = matches.index[0]

    movie_vector = genre_sparse[movie_index]

    similarities = cosine_similarity(
        movie_vector,
        genre_sparse
    ).flatten()

    similar_indices = similarities.argsort()[::-1]

    similar_indices = [
        i for i in similar_indices
        if i != movie_index
    ]

    top_indices = similar_indices[:n]

    recommendations = movies.iloc[
        top_indices
    ][["movieId", "title", "genres"]].copy()

    recommendations["content_score"] = [
        similarities[i]
        for i in top_indices
    ]

    return recommendations

In [153]:
recommend_content(
    "Toy Story (1995)",
    n=10
)

,movieId,title,genres,content_score
70660,227526,The SpongeBob Movie: Sponge on the Run (2020),Adventure|Animation|Children|Comedy|Fantasy,1.0
60252,200614,Wonder Park (2019),Adventure|Animation|Children|Comedy|Fantasy,1.0
9951,33463,DuckTales: The Movie - Treasure of the Lost La...,Adventure|Animation|Children|Comedy|Fantasy,1.0
53494,186177,Trolls Holiday (2017),Adventure|Animation|Children|Comedy|Fantasy,1.0
60260,200630,Missing Link (2019),Adventure|Animation|Children|Comedy|Fantasy,1.0
61889,204188,UglyDolls (2019),Adventure|Animation|Children|Comedy|Fantasy,1.0
69923,225173,Soul (2020),Adventure|Animation|Children|Comedy|Fantasy,1.0
80560,270294,Riverdance: The Animated Adventure (2021),Adventure|Animation|Children|Comedy|Fantasy,1.0
10775,45074,"Wild, The (2006)",Adventure|Animation|Children|Comedy|Fantasy,1.0
63082,206959,Frozen II (2019),Adventure|Animation|Children|Comedy|Fantasy,1.0


In [155]:
selected_movie = "Toy Story (1995)"
content_recs = recommend_content(
    selected_movie,
    n=20
)

In [156]:
collab_recs = recommend_collaborative(
    target_user_id,
    n=20
)

In [157]:
hybrid_recs = pd.merge(
    content_recs[
        ["movieId", "title", "genres", "content_score"]
    ],
    collab_recs[
        ["movieId", "score"]
    ],
    on="movieId",
    how="outer"
)

In [158]:
hybrid_recs["content_score"] = (
    hybrid_recs["content_score"]
    .fillna(0)
)

hybrid_recs["score"] = (
    hybrid_recs["score"]
    .fillna(0)
)

In [159]:
hybrid_recs["hybrid_score"] = (
    0.5 * hybrid_recs["content_score"]
    +
    0.5 * hybrid_recs["score"]
)

In [160]:
hybrid_recs = hybrid_recs.sort_values(
    "hybrid_score",
    ascending=False
)

hybrid_recs[
    ["movieId", "title", "genres", "hybrid_score"]
].head(10)

,movieId,title,genres,hybrid_score
2,162,NaN,NaN,2.5
11,1927,NaN,NaN,2.5
1,28,NaN,NaN,2.5
0,1,NaN,NaN,2.5
4,441,NaN,NaN,2.5
5,866,NaN,NaN,2.5
6,897,NaN,NaN,2.5
3,326,NaN,NaN,2.5
7,1413,NaN,NaN,2.5
8,1484,NaN,NaN,2.5


In [161]:
def recommend_hybrid(movie_title, user_id, n=10, alpha=0.5):

    # Content-based recommendations
    content_recs = recommend_content(
        movie_title,
        n=30
    )

    # Collaborative recommendations
    collab_recs = recommend_collaborative(
        user_id,
        n=30
    )

    # Keep only the columns we need
    content_recs = content_recs[
        ["movieId", "title", "genres", "content_score"]
    ]

    collab_recs = collab_recs[
        ["movieId", "score"]
    ]

    # Combine both recommenders
    hybrid = pd.merge(
        content_recs,
        collab_recs,
        on="movieId",
        how="outer"
    )

    # Fill missing scores
    hybrid["content_score"] = (
        hybrid["content_score"].fillna(0)
    )

    hybrid["score"] = (
        hybrid["score"].fillna(0)
    )

    # Calculate hybrid score
    hybrid["hybrid_score"] = (
        alpha * hybrid["content_score"]
        +
        (1 - alpha) * hybrid["score"]
    )

    # Sort recommendations
    hybrid = hybrid.sort_values(
        "hybrid_score",
        ascending=False
    )

    return hybrid[
        [
            "movieId",
            "title",
            "genres",
            "content_score",
            "score",
            "hybrid_score"
        ]
    ].head(n)

In [162]:
hybrid_recommendations = recommend_hybrid(
    "Toy Story (1995)",
    target_user_id,
    n=10
)

hybrid_recommendations

,movieId,title,genres,content_score,score,hybrid_score
2,162,NaN,NaN,0.0,5.0,2.5
12,1927,NaN,NaN,0.0,5.0,2.5
1,28,NaN,NaN,0.0,5.0,2.5
0,1,NaN,NaN,0.0,5.0,2.5
4,441,NaN,NaN,0.0,5.0,2.5
5,866,NaN,NaN,0.0,5.0,2.5
6,897,NaN,NaN,0.0,5.0,2.5
7,1413,NaN,NaN,0.0,5.0,2.5
8,1484,NaN,NaN,0.0,5.0,2.5
9,1594,NaN,NaN,0.0,5.0,2.5


In [163]:
recommend_hybrid(
    "Toy Story (1995)",
    target_user_id,
    n=10,
    alpha=0.7
)

,movieId,title,genres,content_score,score,hybrid_score
2,162,NaN,NaN,0.0,5.0,1.5
12,1927,NaN,NaN,0.0,5.0,1.5
1,28,NaN,NaN,0.0,5.0,1.5
0,1,NaN,NaN,0.0,5.0,1.5
4,441,NaN,NaN,0.0,5.0,1.5
5,866,NaN,NaN,0.0,5.0,1.5
6,897,NaN,NaN,0.0,5.0,1.5
7,1413,NaN,NaN,0.0,5.0,1.5
8,1484,NaN,NaN,0.0,5.0,1.5
9,1594,NaN,NaN,0.0,5.0,1.5


In [164]:
def min_max_normalize(series):
    min_value = series.min()
    max_value = series.max()

    if max_value == min_value:
        return pd.Series(
            0.0,
            index=series.index
        )

    return (
        (series - min_value)
        / (max_value - min_value)
    )

In [165]:
def recommend_hybrid(
    movie_title,
    user_id,
    n=10,
    alpha=0.5
):

    content_recs = recommend_content(
        movie_title,
        n=30
    )

    collab_recs = recommend_collaborative(
        user_id,
        n=30
    )

    content_recs = content_recs[
        ["movieId", "title", "genres", "content_score"]
    ]

    collab_recs = collab_recs[
        ["movieId", "score"]
    ]

    hybrid = pd.merge(
        content_recs,
        collab_recs,
        on="movieId",
        how="outer"
    )

    hybrid["content_score"] = (
        hybrid["content_score"].fillna(0)
    )

    hybrid["score"] = (
        hybrid["score"].fillna(0)
    )

    # Normalize both scores
    hybrid["content_normalized"] = (
        min_max_normalize(
            hybrid["content_score"]
        )
    )

    hybrid["collab_normalized"] = (
        min_max_normalize(
            hybrid["score"]
        )
    )

    # Hybrid score
    hybrid["hybrid_score"] = (
        alpha * hybrid["content_normalized"]
        +
        (1 - alpha) * hybrid["collab_normalized"]
    )

    hybrid = hybrid.sort_values(
        "hybrid_score",
        ascending=False
    )

    return hybrid[
        [
            "movieId",
            "title",
            "genres",
            "content_score",
            "score",
            "hybrid_score"
        ]
    ].head(n)

In [166]:
hybrid_recommendations = recommend_hybrid(
    "Toy Story (1995)",
    target_user_id,
    n=10
)

hybrid_recommendations

,movieId,title,genres,content_score,score,hybrid_score
2,162,NaN,NaN,0.0,5.0,0.5
12,1927,NaN,NaN,0.0,5.0,0.5
57,276403,Chickenhare and the Hamster of Darkness (2022),Adventure|Animation|Children|Comedy|Fantasy,1.0,0.0,0.5
56,270294,Riverdance: The Animated Adventure (2021),Adventure|Animation|Children|Comedy|Fantasy,1.0,0.0,0.5
51,266324,Casper's Scare School (2006),Adventure|Animation|Children|Comedy|Fantasy,1.0,0.0,0.5
50,247988,Luca (2021),Adventure|Animation|Children|Comedy|Fantasy,1.0,0.0,0.5
49,227526,The SpongeBob Movie: Sponge on the Run (2020),Adventure|Animation|Children|Comedy|Fantasy,1.0,0.0,0.5
48,225173,Soul (2020),Adventure|Animation|Children|Comedy|Fantasy,1.0,0.0,0.5
47,215229,Trolls World Tour (2020),Adventure|Animation|Children|Comedy|Fantasy,1.0,0.0,0.5
46,206959,Frozen II (2019),Adventure|Animation|Children|Comedy|Fantasy,1.0,0.0,0.5


In [167]:
recommend_hybrid(
    "Toy Story (1995)",
    target_user_id,
    n=10,
    alpha=0.7
)

,movieId,title,genres,content_score,score,hybrid_score
55,269152,Pil's Adventures (2021),Adventure|Animation|Children|Comedy|Fantasy,1.0,0.0,0.7
54,269122,Bunyan and Babe (2017),Adventure|Animation|Children|Comedy|Fantasy,1.0,0.0,0.7
57,276403,Chickenhare and the Hamster of Darkness (2022),Adventure|Animation|Children|Comedy|Fantasy,1.0,0.0,0.7
56,270294,Riverdance: The Animated Adventure (2021),Adventure|Animation|Children|Comedy|Fantasy,1.0,0.0,0.7
51,266324,Casper's Scare School (2006),Adventure|Animation|Children|Comedy|Fantasy,1.0,0.0,0.7
50,247988,Luca (2021),Adventure|Animation|Children|Comedy|Fantasy,1.0,0.0,0.7
49,227526,The SpongeBob Movie: Sponge on the Run (2020),Adventure|Animation|Children|Comedy|Fantasy,1.0,0.0,0.7
48,225173,Soul (2020),Adventure|Animation|Children|Comedy|Fantasy,1.0,0.0,0.7
47,215229,Trolls World Tour (2020),Adventure|Animation|Children|Comedy|Fantasy,1.0,0.0,0.7
46,206959,Frozen II (2019),Adventure|Animation|Children|Comedy|Fantasy,1.0,0.0,0.7


In [168]:
recommend_hybrid(
    "Toy Story (1995)",
    target_user_id,
    n=10,
    alpha=0.3
)

,movieId,title,genres,content_score,score,hybrid_score
2,162,NaN,NaN,0.0,5.0,0.7
12,1927,NaN,NaN,0.0,5.0,0.7
1,28,NaN,NaN,0.0,5.0,0.7
0,1,NaN,NaN,0.0,5.0,0.7
4,441,NaN,NaN,0.0,5.0,0.7
5,866,NaN,NaN,0.0,5.0,0.7
6,897,NaN,NaN,0.0,5.0,0.7
7,1413,NaN,NaN,0.0,5.0,0.7
8,1484,NaN,NaN,0.0,5.0,0.7
9,1594,NaN,NaN,0.0,5.0,0.7


In [170]:
movie_index_lookup = pd.Series(
    movies.index,
    index=movies["movieId"]
)
movie_index_lookup.head()

movieId
1    0
2    1
3    2
4    3
5    4
dtype: int64

In [171]:
def recommend_content_fast(movie_title, n=10):

    matches = movies[
        movies["title"] == movie_title
    ]

    if matches.empty:
        return pd.DataFrame()

    movie_index = matches.index[0]

    movie_vector = genre_sparse[movie_index]

    similarities = cosine_similarity(
        movie_vector,
        genre_sparse
    ).flatten()

    similarities[movie_index] = -1

    top_indices = similarities.argsort()[
        -n:
    ][::-1]

    recommendations = movies.iloc[
        top_indices
    ][
        ["movieId", "title", "genres"]
    ].copy()

    recommendations["content_score"] = (
        similarities[top_indices]
    )

    return recommendations

In [172]:
recommend_content_fast(
    "Toy Story (1995)",
    n=10
)

,movieId,title,genres,content_score
53494,186177,Trolls Holiday (2017),Adventure|Animation|Children|Comedy|Fantasy,1.0
53485,186159,Tangled: Before Ever After (2017),Adventure|Animation|Children|Comedy|Fantasy,1.0
2203,2294,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy,1.0
85629,286131,The Super Mario Bros. Movie (2023),Adventure|Animation|Children|Comedy|Fantasy,1.0
17436,91355,Asterix and the Vikings (Astérix et les Viking...,Adventure|Animation|Children|Comedy|Fantasy,1.0
66530,215229,Trolls World Tour (2020),Adventure|Animation|Children|Comedy|Fantasy,1.0
22298,114240,Aladdin (1992),Adventure|Animation|Children|Comedy|Fantasy,1.0
3654,3754,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.0
58919,197743,Here Comes the Grump (2018),Adventure|Animation|Children|Comedy|Fantasy,1.0
3913,4016,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.0


In [175]:
from pathlib import Path

print(Path.cwd())

c:\Users\radhe\OneDrive\Desktop\movie-recommendation-system\notebooks


In [176]:
import sys
from pathlib import Path

project_root = Path.cwd().parent

sys.path.append(str(project_root))

print(project_root)

c:\Users\radhe\OneDrive\Desktop\movie-recommendation-system


In [178]:
print((project_root / "src").exists())

True


In [182]:
from src.data_loader import load_data

In [184]:
print(src.data_loader.__file__)

NameError: name 'src' is not defined

In [185]:
import src.data_loader

In [186]:
import importlib

importlib.reload(src.data_loader)

<module 'src.data_loader' from 'c:\\Users\\radhe\\OneDrive\\Desktop\\movie-recommendation-system\\src\\data_loader.py'>

In [187]:
import inspect

print(inspect.getsource(src.data_loader.load_data))

def load_data():

    movies_path = PROJECT_ROOT / "data" / "ml-32m" / "movies.csv"
    ratings_path = PROJECT_ROOT / "data" / "ml-32m" / "ratings.csv"

    movies = pd.read_csv(movies_path)

    ratings = pd.read_csv(ratings_path)

    return movies, ratings



In [188]:
movies, ratings = src.data_loader.load_data()

print(movies.shape)
print(ratings.shape)

(87585, 3)
(32000204, 4)


In [189]:
from src.content_model import build_content_model

genre_sparse = build_content_model(movies)

print(genre_sparse.shape)

(87585, 20)


In [190]:
from src.content_model import recommend_content

content_test = recommend_content(
    "Toy Story (1995)",
    movies,
    genre_sparse,
    n=10
)

content_test

,movieId,title,genres,content_score
53494,186177,Trolls Holiday (2017),Adventure|Animation|Children|Comedy|Fantasy,1.0
53485,186159,Tangled: Before Ever After (2017),Adventure|Animation|Children|Comedy|Fantasy,1.0
2203,2294,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy,1.0
85629,286131,The Super Mario Bros. Movie (2023),Adventure|Animation|Children|Comedy|Fantasy,1.0
17436,91355,Asterix and the Vikings (Astérix et les Viking...,Adventure|Animation|Children|Comedy|Fantasy,1.0
66530,215229,Trolls World Tour (2020),Adventure|Animation|Children|Comedy|Fantasy,1.0
22298,114240,Aladdin (1992),Adventure|Animation|Children|Comedy|Fantasy,1.0
3654,3754,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.0
58919,197743,Here Comes the Grump (2018),Adventure|Animation|Children|Comedy|Fantasy,1.0
3913,4016,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.0


In [191]:
from src.collaborative_model import (
    build_user_item_matrix
)

In [192]:
(
    user_item_sparse,
    user_ids,
    movie_ids,
    user_id_to_index,
    movie_id_to_index
) = build_user_item_matrix(ratings)

In [193]:
print(user_item_sparse.shape)
print(user_item_sparse.nnz)

(200948, 84432)
32000204


In [195]:
target_user_id = current_user_id
from src.collaborative_model import recommend_collaborative

collab_test = recommend_collaborative(
    target_user_id,
    ratings,
    user_item_sparse,
    user_ids,
    user_id_to_index,
    movies,
    n=10
)

collab_test

,movieId,title,genres,score,rating_count
0,162,Crumb (1994),Documentary,5.0,3
1,1927,All Quiet on the Western Front (1930),Action|Drama|War,5.0,2
2,28,Persuasion (1995),Drama|Romance,5.0,1
3,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,5.0,2
4,3035,Mister Roberts (1955),Comedy|Drama|War,5.0,1
5,2859,Stop Making Sense (1984),Documentary|Musical,5.0,1
6,2948,From Russia with Love (1963),Action|Adventure|Thriller,5.0,1
7,3578,Gladiator (2000),Action|Adventure|Drama,5.0,1
8,2670,Run Silent Run Deep (1958),War,5.0,1
9,2628,Star Wars: Episode I - The Phantom Menace (1999),Action|Adventure|Sci-Fi,5.0,2


In [206]:
import importlib
import src.hybrid_model

importlib.reload(src.hybrid_model)


<module 'src.hybrid_model' from 'c:\\Users\\radhe\\OneDrive\\Desktop\\movie-recommendation-system\\src\\hybrid_model.py'>

In [207]:
from src.hybrid_model import recommend_hybrid

In [208]:
hybrid_test = recommend_hybrid(
    movie_title="Toy Story (1995)",
    user_id=current_user_id,
    movies=movies,
    ratings=ratings,
    genre_sparse=genre_sparse,
    user_item_sparse=user_item_sparse,
    user_ids=user_ids,
    user_id_to_index=user_id_to_index,
    n=10,
    alpha=0.5
)

hybrid_test

,movieId,title,genres,content_score,score,hybrid_score
2,162,Crumb (1994),Documentary,0.0,5.0,0.5
12,1927,All Quiet on the Western Front (1930),Action|Drama|War,0.0,5.0,0.5
22,2294,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy,1.0,0.0,0.5
56,270294,Riverdance: The Animated Adventure (2021),Adventure|Animation|Children|Comedy|Fantasy,1.0,0.0,0.5
51,228513,The Snow Queen: Mirror Lands (2018),Adventure|Animation|Children|Comedy|Fantasy,1.0,0.0,0.5
50,215229,Trolls World Tour (2020),Adventure|Animation|Children|Comedy|Fantasy,1.0,0.0,0.5
49,213207,Onward (2020),Adventure|Animation|Children|Comedy|Fantasy,1.0,0.0,0.5
48,197743,Here Comes the Grump (2018),Adventure|Animation|Children|Comedy|Fantasy,1.0,0.0,0.5
47,192981,Penguin Highway (2018),Adventure|Animation|Children|Comedy|Fantasy,1.0,0.0,0.5
54,267944,Christmas in Tattertown (1988),Adventure|Animation|Children|Comedy|Fantasy,1.0,0.0,0.5
